# Dynamics 365 CRM Account Ingestion

This notebook ingests account data from **Dynamics 365 CRM (Reliance Nigeria)** into the Fabric Lakehouse.

## Target Schema
- **Name** - Account name
- **Relationship Type** - Customer, Partner, Prospect, etc.
- **Owner** - Account owner's full name
- **Territory** - Sales territory assignment
- **Status** - Active/Inactive

## Data Flow
1. Authenticate to Dynamics 365 CRM
2. Query active accounts from the `account` table
3. Enrich with owner and territory lookups
4. Transform to target schema
5. Save to Lakehouse Delta table `crm_accounts`
6. Generate summary statistics

In [ ]:
# Diagnostic 3: can /beta/tenants return MORE than the 15 (baseline / assessment-only / unlicensed)?
import requests, json

INF_BASE = "https://api-us.inforcer.com/api"
INF_KEY = "0ce9a0ed6d1e4df998e62e57633a4f17"
H = {"Inf-Api-Key": INF_KEY}

def count_tenants(params=None):
    r = requests.get(f"{INF_BASE}/beta/tenants", headers=H, params=params)
    if not r.ok:
        return f"HTTP {r.status_code}"
    d = r.json().get("data", [])
    names = sorted({t.get("tenantDnsName") for t in d})
    return f"{len(d)} tenants"

variants = [
    None,
    {"isBaseline": "true"},
    {"includeBaselines": "true"},
    {"includeBaseline": "true"},
    {"type": "all"},
    {"status": "all"},
    {"licensed": "false"},
    {"includeUnlicensed": "true"},
    {"all": "true"},
    {"pageSize": 500},
]
for v in variants:
    print(f"{str(v):35s} -> {count_tenants(v)}")

# Try a few alternative endpoints that might list assessment-only / all tenants
for ep in ["/beta/tenants/all", "/beta/assessments/tenants", "/beta/tenantAssessments", "/beta/tenants/assessments"]:
    r = requests.get(f"{INF_BASE}{ep}", headers=H)
    print(f"GET {ep} -> HTTP {r.status_code} {('items='+str(len(r.json().get('data', [])))) if r.ok and isinstance(r.json(), dict) else ''}")


StatementMeta(, 11f5875e-9aba-47ae-82ab-0ee7ac726f83, 8, Finished, Available, Finished, False)

None                                -> 15 tenants
{'isBaseline': 'true'}              -> 15 tenants
{'includeBaselines': 'true'}        -> 15 tenants
{'includeBaseline': 'true'}         -> 15 tenants
{'type': 'all'}                     -> 15 tenants
{'status': 'all'}                   -> 15 tenants
{'licensed': 'false'}               -> 15 tenants
{'includeUnlicensed': 'true'}       -> 15 tenants
{'all': 'true'}                     -> 15 tenants
{'pageSize': 500}                   -> 15 tenants
GET /beta/tenants/all -> HTTP 400 
GET /beta/assessments/tenants -> HTTP 404 
GET /beta/tenantAssessments -> HTTP 404 
GET /beta/tenants/assessments -> HTTP 400 


In [41]:
# Install Dataverse SDK for Python
%pip install PowerPlatform-Dataverse-client azure-identity --quiet

StatementMeta(, 2c2d2f5b-c336-4dfe-abbf-af9295aaa05a, 9, Finished, Available, Finished, False)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fsspec-wrapper 0.1.15 requires PyJWT>=2.6.0, but you have pyjwt 2.4.0 which is incompatible.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [42]:
from azure.identity import ClientSecretCredential
from PowerPlatform.Dataverse.client import DataverseClient
from PowerPlatform.Dataverse.models.filters import col
import pandas as pd
from datetime import datetime

# ─────────────────────────────────────────────────────────────────────────────
# Dynamics 365 CRM Configuration - Reliance Nigeria
# ─────────────────────────────────────────────────────────────────────────────
# Nigeria environment URL (Europe region): https://{org}.crm4.dynamics.com
CRM_BASE_URL = "https://relianceinfo.crm4.dynamics.com"   # Reliance Nigeria CRM

# ── Service Principal (app registration: "Dynamics365Accounts-Fabric Sync") ──
TENANT_ID = "0b60fed4-5fc9-409d-95f2-271114f4c86f"        # relianceinfosystems.com
CLIENT_ID = "1fd98ce0-95a2-468f-98f3-cca5ccc6dc12"        # Application (Client) ID

# ── Client secret is stored in Azure Key Vault (NEVER hardcode secrets) ──
KEY_VAULT_URL = "https://dynamicsfabricsynckey.vault.azure.net/"
SECRET_NAME   = "dynamics365-fabric-sync-secret"

# Lakehouse target
TARGET_TABLE_NAME = "crm_accounts"
LAKEHOUSE_PATH = f"Tables/{TARGET_TABLE_NAME}"

print(f"🔗 Connecting to Dynamics 365 CRM: {CRM_BASE_URL}")
print(f"   Tenant: {TENANT_ID}")
print(f"   Client: {CLIENT_ID}")


StatementMeta(, 2c2d2f5b-c336-4dfe-abbf-af9295aaa05a, 11, Finished, Available, Finished, False)

🔗 Connecting to Dynamics 365 CRM: https://relianceinfo.crm4.dynamics.com
   Tenant: 0b60fed4-5fc9-409d-95f2-271114f4c86f
   Client: 1fd98ce0-95a2-468f-98f3-cca5ccc6dc12


In [43]:
# ─────────────────────────────────────────────────────────────────────────────
# Authenticate to Dynamics 365 using a Service Principal (client credentials)
# ─────────────────────────────────────────────────────────────────────────────
# Fabric notebooks do NOT support DefaultAzureCredential (no IMDS/managed-identity
# endpoint in the Spark runtime). We authenticate with an Entra app registration
# whose client secret is retrieved securely from Azure Key Vault at runtime.
# The app is registered as an Application User in the Nigeria Dynamics 365
# environment with the "Accounts Manager" security role (read access to accounts).

print("🔐 Authenticating to Dynamics 365 with service principal...")

# Retrieve the client secret from Key Vault (never hardcode it in the notebook)
client_secret = notebookutils.credentials.getSecret(KEY_VAULT_URL, SECRET_NAME)

credential = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=client_secret,
)

crm_client = DataverseClient(base_url=CRM_BASE_URL, credential=credential)

# Force a real token acquisition now so auth issues surface here (the client is lazy)
_ = credential.get_token(f"{CRM_BASE_URL}/.default")

print("✅ Successfully authenticated to Dynamics 365 CRM")


StatementMeta(, 2c2d2f5b-c336-4dfe-abbf-af9295aaa05a, 12, Finished, Available, Finished, False)

🔐 Authenticating to Dynamics 365 with service principal...
✅ Successfully authenticated to Dynamics 365 CRM


In [ ]:
# Query CRM Accounts with the required fields
print("📊 Querying CRM accounts...")

# Field names in Dynamics 365:
# - name: Account name
# - customertypecode: Customer/Relationship type (picklist)
# - ownerid: Owner / account manager (lookup to systemuser)
# - territoryid: Territory (lookup to territory table)
# - statecode: Status (Active=0, Inactive=1)
# - statuscode: Detailed status code

# Page through ALL accounts using execute_pages() so we don't miss any records
# regardless of how large the CRM org grows. Dataverse caps each response page
# (default ~5000 rows), so we explicitly stream page-by-page and concatenate.
PAGE_SIZE = 2000

try:
    pages = []
    total = 0
    for page_num, page in enumerate(
        crm_client.query.builder("account")
        .select(
            "accountid",
            "name",
            "customertypecode",           # Relationship type
            "statecode",                  # Status (0=Active, 1=Inactive)
            "statuscode",                 # Detailed status
            "_ownerid_value",             # Owner / account manager ID
            "_territoryid_value",         # Territory ID (if used)
            "_primarycontactid_value"     # Primary contact ID (for Email (Primary Contact))
        )
        .where(col("statecode") == 0)     # Filter for active accounts only
        .order_by("name")
        .page_size(PAGE_SIZE)
        .execute_pages()
    ):
        page_df = page.to_dataframe()
        pages.append(page_df)
        total += len(page_df)
        print(f"   📄 Page {page_num + 1}: {len(page_df)} accounts (running total: {total})")

    accounts_df = pd.concat(pages, ignore_index=True) if pages else pd.DataFrame()

    print(f"✅ Retrieved {len(accounts_df)} active accounts from CRM")

    # Display sample
    if len(accounts_df) > 0:
        print("\n📋 Sample of raw data:")
        print(accounts_df.head(3))

except Exception as e:
    print(f"❌ Error querying accounts: {e}")
    raise

StatementMeta(, 2c2d2f5b-c336-4dfe-abbf-af9295aaa05a, 13, Finished, Available, Finished, False)

📊 Querying CRM accounts...
   📄 Page 1: 2000 accounts (running total: 2000)
   📄 Page 2: 2000 accounts (running total: 4000)
   📄 Page 3: 2000 accounts (running total: 6000)
   📄 Page 4: 2000 accounts (running total: 8000)
   📄 Page 5: 2000 accounts (running total: 10000)
   📄 Page 6: 2000 accounts (running total: 12000)
   📄 Page 7: 2000 accounts (running total: 14000)
   📄 Page 8: 2000 accounts (running total: 16000)
   📄 Page 9: 2000 accounts (running total: 18000)
   📄 Page 10: 2000 accounts (running total: 20000)
   📄 Page 11: 2000 accounts (running total: 22000)
   📄 Page 12: 55 accounts (running total: 22055)
✅ Retrieved 22055 active accounts from CRM

📋 Sample of raw data:
                                                name  statuscode  \
0                                      BST Solutions           1   
1                                      BST Solutions           1   
2   Ideal Industrial Distributors, Echo Marine Lt...           1   

                              account

In [ ]:
# Enrich data with Owner (account manager), Territory, and Primary Contact email
print("\n🔍 Enriching data with owner, territory, and primary contact information...")

LOOKUP_PAGE_SIZE = 2000

def fetch_all(table, columns):
    """Page through an entire Dataverse table and return a single DataFrame."""
    frames = []
    for page in (
        crm_client.query.builder(table)
        .select(*columns)
        .page_size(LOOKUP_PAGE_SIZE)
        .execute_pages()
    ):
        frames.append(page.to_dataframe())
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)

# ── Owners (account managers): name + email ──────────────────────────────────
# We page through all system users once and build lookup maps. The owner's name
# AND email let the seller team reliably correlate an Inforcer customer to the
# responsible seller.
if '_ownerid_value' in accounts_df.columns:
    print("   Fetching system users (account managers)...")
    owners_df = fetch_all("systemuser", ["systemuserid", "fullname", "internalemailaddress"])

    if not owners_df.empty:
        owner_name_lookup = dict(zip(owners_df['systemuserid'], owners_df['fullname']))
        email_series = owners_df['internalemailaddress'] if 'internalemailaddress' in owners_df.columns else pd.Series(dtype=str)
        owner_email_lookup = dict(zip(owners_df['systemuserid'], email_series))

        accounts_df['Owner'] = accounts_df['_ownerid_value'].map(owner_name_lookup).fillna("Unassigned")
        accounts_df['OwnerEmail'] = accounts_df['_ownerid_value'].map(owner_email_lookup).fillna("")
        print(f"   ✅ Loaded {len(owner_name_lookup)} users; mapped owners for {len(accounts_df)} accounts")
    else:
        accounts_df['Owner'] = "Unassigned"
        accounts_df['OwnerEmail'] = ""
else:
    accounts_df['Owner'] = "Unassigned"
    accounts_df['OwnerEmail'] = ""

# ── Territories (optional in some orgs) ──────────────────────────────────────
if '_territoryid_value' in accounts_df.columns and accounts_df['_territoryid_value'].notna().any():
    print("   Fetching territories...")
    try:
        territories_df = fetch_all("territory", ["territoryid", "name"])
        territory_lookup = dict(zip(territories_df['territoryid'], territories_df['name']))
        accounts_df['Territory'] = accounts_df['_territoryid_value'].map(territory_lookup).fillna("Not Assigned")
        print(f"   ✅ Mapped {len(territory_lookup)} territory names")
    except Exception as e:
        print(f"   ⚠️ Could not fetch territories (table may not exist): {e}")
        accounts_df['Territory'] = "Not Assigned"
else:
    accounts_df['Territory'] = "Not Assigned"

# ── Primary contact email — "Email (Primary Contact)" ────────────────────────
# Each account's primarycontactid points to a contact whose emailaddress1 is the
# CRM "Email (Primary Contact)" column. The email DOMAIN is the strongest key for
# correlating a CRM account to an Inforcer tenant (inforcer_tenants.tenantDnsName).
if '_primarycontactid_value' in accounts_df.columns and accounts_df['_primarycontactid_value'].notna().any():
    print("   Fetching primary contacts (Email (Primary Contact))...")
    try:
        contacts_df = fetch_all("contact", ["contactid", "emailaddress1"])
        contact_email_lookup = dict(zip(contacts_df['contactid'], contacts_df['emailaddress1']))
        accounts_df['PrimaryContactEmail'] = accounts_df['_primarycontactid_value'].map(contact_email_lookup).fillna("")
        mapped = (accounts_df['PrimaryContactEmail'] != "").sum()
        print(f"   ✅ Loaded {len(contact_email_lookup)} contacts; mapped emails for {mapped} accounts")
    except Exception as e:
        print(f"   ⚠️ Could not fetch contacts: {e}")
        accounts_df['PrimaryContactEmail'] = ""
else:
    accounts_df['PrimaryContactEmail'] = ""

print("✅ Data enrichment complete")


StatementMeta(, 2c2d2f5b-c336-4dfe-abbf-af9295aaa05a, 15, Finished, Available, Finished, False)


🔍 Enriching data with owner, territory, and primary contact information...
   Fetching system users (account managers)...
   ✅ Loaded 1001 users; mapped owners for 22055 accounts
   Fetching territories...
   ✅ Mapped 16 territory names
   Fetching primary contacts (Email (Primary Contact))...
   ✅ Loaded 1174740 contacts; mapped emails for 1301 accounts
✅ Data enrichment complete


In [ ]:
# Transform to final schema
print("\n🔄 Transforming to target schema...")

# Map customertypecode picklist values to readable labels
# Common Dynamics 365 customer type codes:
# 1 = Competitor, 2 = Consultant, 3 = Customer, 4 = Investor, 5 = Partner, 
# 6 = Influencer, 7 = Press, 8 = Prospect, 9 = Reseller, 10 = Supplier, 11 = Vendor, 12 = Other
customer_type_mapping = {
    1: "Competitor",
    2: "Consultant", 
    3: "Customer",
    4: "Investor",
    5: "Partner",
    6: "Influencer",
    7: "Press",
    8: "Prospect",
    9: "Reseller",
    10: "Supplier",
    11: "Vendor",
    12: "Other"
}

# Map status codes to readable labels
status_mapping = {
    0: "Active",
    1: "Inactive"
}

# Create final dataframe with desired schema
crm_accounts_final = pd.DataFrame({
    'AccountID': accounts_df['accountid'],
    'Name': accounts_df['name'],
    'RelationshipType': accounts_df['customertypecode'].map(customer_type_mapping).fillna("Unknown"),
    'Owner': accounts_df['Owner'],
    'OwnerEmail': accounts_df['OwnerEmail'],
    'PrimaryContactEmail': accounts_df['PrimaryContactEmail'],
    'Territory': accounts_df['Territory'],
    'Status': accounts_df['statecode'].map(status_mapping).fillna("Unknown"),
    'LastSyncDate': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
})

print(f"✅ Transformed {len(crm_accounts_final)} records")
print(f"\n📊 Final schema:")
print(crm_accounts_final.info())
print(f"\n📋 Sample data:")
print(crm_accounts_final.head(10))

StatementMeta(, 2c2d2f5b-c336-4dfe-abbf-af9295aaa05a, 16, Finished, Available, Finished, False)


🔄 Transforming to target schema...
✅ Transformed 22055 records

📊 Final schema:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22055 entries, 0 to 22054
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   AccountID            22055 non-null  object
 1   Name                 22055 non-null  object
 2   RelationshipType     22055 non-null  object
 3   Owner                22055 non-null  object
 4   OwnerEmail           22055 non-null  object
 5   PrimaryContactEmail  22055 non-null  object
 6   Territory            22055 non-null  object
 7   Status               22055 non-null  object
 8   LastSyncDate         22055 non-null  object
dtypes: object(9)
memory usage: 1.5+ MB
None

📋 Sample data:
                              AccountID  \
0  89d920a8-5aae-f011-bbd2-6045bd98b330   
1  b722b773-5aae-f011-bbd2-7ced8d906675   
2  9a998f22-9831-f011-8c4e-6045bd9b6920   
3  b9655ed6-192d-ec11-b6e5-000d3a2aac76

In [ ]:
# Save to Fabric Lakehouse
print(f"\n💾 Saving to Fabric Lakehouse table '{TARGET_TABLE_NAME}'...")

# Explicit OneLake path to the lakehouse Tables folder so the write does not
# depend on the live session having a default lakehouse attached.
# ManagedServiceData is a schema-enabled lakehouse (default schema = dbo), so we
# write under Tables/dbo/ to register crm_accounts as a catalog table that is
# visible in the Lakehouse explorer and queryable via the SQL endpoint.
LAKEHOUSE_ID = "3d0144b0-12bf-4483-9508-67b26b1fd125"   # ManagedServiceData
WORKSPACE_ID = "0f895a7e-09c6-4645-8b47-d272bc687b8a"   # SEManagedService
SCHEMA_NAME = "dbo"
TABLE_ABFSS_PATH = (
    f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
    f"{LAKEHOUSE_ID}/Tables/{SCHEMA_NAME}/{TARGET_TABLE_NAME}"
)

try:
    # Convert to Spark DataFrame
    spark_df = spark.createDataFrame(crm_accounts_final)

    # Write to Delta table in the lakehouse
    # Mode 'overwrite' replaces the entire table on each run
    # Use 'append' to add new records, or implement upsert logic for updates
    spark_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(TABLE_ABFSS_PATH)

    print(f"✅ Successfully saved {len(crm_accounts_final)} CRM accounts to lakehouse table '{TARGET_TABLE_NAME}'")
    print(f"   Table location: {TABLE_ABFSS_PATH}")

    # Verify the save
    record_count = spark.read.format("delta").load(TABLE_ABFSS_PATH).count()
    print(f"✅ Verification: Table now contains {record_count} records")

except Exception as e:
    print(f"❌ Error saving to lakehouse: {e}")
    raise

print("\n🎉 CRM account ingestion complete!")

StatementMeta(, 2c2d2f5b-c336-4dfe-abbf-af9295aaa05a, 17, Finished, Available, Finished, False)


💾 Saving to Fabric Lakehouse table 'crm_accounts'...
✅ Successfully saved 22055 CRM accounts to lakehouse table 'crm_accounts'
   Table location: abfss://0f895a7e-09c6-4645-8b47-d272bc687b8a@onelake.dfs.fabric.microsoft.com/3d0144b0-12bf-4483-9508-67b26b1fd125/Tables/dbo/crm_accounts
✅ Verification: Table now contains 22055 records

🎉 CRM account ingestion complete!


In [21]:
# Optional: View summary statistics and data quality
print("\n📈 Data Summary:")
print("=" * 60)

# Summary by relationship type
print("\n📊 Accounts by Relationship Type:")
relationship_summary = crm_accounts_final['RelationshipType'].value_counts()
print(relationship_summary)

# Summary by owner
print("\n👤 Accounts by Owner (Top 10):")
owner_summary = crm_accounts_final['Owner'].value_counts().head(10)
print(owner_summary)

# Summary by territory
print("\n🗺️ Accounts by Territory:")
territory_summary = crm_accounts_final['Territory'].value_counts()
print(territory_summary)

# Summary by status
print("\n📍 Accounts by Status:")
status_summary = crm_accounts_final['Status'].value_counts()
print(status_summary)

print("\n" + "=" * 60)
print("✅ All statistics generated successfully")

StatementMeta(, 736a8729-f46c-40d7-a51e-5967f0a19b0f, 22, Finished, Available, Finished, False)


📈 Data Summary:

📊 Accounts by Relationship Type:
RelationshipType
Unknown       14931
Prospect       5027
Customer       1841
Vendor          164
Reseller         62
Consultant       11
Competitor        9
Partner           4
Supplier          3
Other             3
Name: count, dtype: int64

👤 Accounts by Owner (Top 10):
Owner
Donald Izuchukwu     6558
Yemi Popoola         4076
Ayodele Akinwunmi    2794
Abimbola Waheed       761
Olamide Azeez         744
Femi Ipinlaye         636
Udeme Richard         616
Jumoke Aliu           607
Mary Enayigbemu       596
Sola Jinadu           545
Name: count, dtype: int64

🗺️ Accounts by Territory:
Territory
Not Assigned                   13889
Reliance Nigeria                6775
Reliance Botswana                370
Reliance Datatech                302
Reliance Egypt                   300
Cloudware (Ghana)                133
Reliance Pakistan                103
Reliance South Africa             49
Reliance UK                       48
Reliance Nami

## Correlate Inforcer tenants ↔ CRM accounts (by email domain)

Match each **Inforcer tenant** (`dbo.inforcer_tenants.tenantDnsName`) to its **CRM account**
using the domain of the account's *Email (Primary Contact)* (`dbo.crm_accounts.PrimaryContactEmail`).
This identifies the **responsible seller/owner** for every customer we are assessing.
The result is written to `dbo.inforcer_crm_correlation`.


In [ ]:
# Correlate Inforcer tenants with CRM accounts via email domain
from pyspark.sql import functions as F
from datetime import datetime

print("🔗 Correlating Inforcer tenants with CRM accounts by email domain...")

# Self-contained so this cell can run independently of the ingestion cells above.
_WORKSPACE_ID = "0f895a7e-09c6-4645-8b47-d272bc687b8a"   # SEManagedService
_LAKEHOUSE_ID = "3d0144b0-12bf-4483-9508-67b26b1fd125"   # ManagedServiceData
_TABLES = (
    f"abfss://{_WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{_LAKEHOUSE_ID}/Tables/dbo"
)

# ── Load CRM accounts and derive the primary-contact email domain ────────────
crm = spark.read.format("delta").load(f"{_TABLES}/crm_accounts")
crm_dom = (
    crm.where((F.col("PrimaryContactEmail").isNotNull()) & (F.col("PrimaryContactEmail") != ""))
       .withColumn("crm_domain", F.lower(F.trim(F.element_at(F.split("PrimaryContactEmail", "@"), -1))))
       .where(F.col("crm_domain") != "")
)

# ── Load Inforcer tenants and normalise their DNS (domain) name ──────────────
tenants = spark.read.format("delta").load(f"{_TABLES}/inforcer_tenants")
tenants_dom = tenants.withColumn("tenant_domain", F.lower(F.trim(F.col("tenantDnsName"))))

# ── Left-join from the tenant side so every Inforcer tenant is represented ───
correlation = (
    tenants_dom.alias("t")
    .join(crm_dom.alias("c"), F.col("t.tenant_domain") == F.col("c.crm_domain"), "left")
    .select(
        F.col("t.clientTenantId").alias("InforcerClientTenantId"),
        F.col("t.msTenantId").alias("MsTenantId"),
        F.col("t.tenantFriendlyName").alias("InforcerFriendlyName"),
        F.col("t.tenantDnsName").alias("TenantDomain"),
        F.col("c.AccountID").alias("CrmAccountID"),
        F.col("c.Name").alias("CrmAccountName"),
        F.col("c.RelationshipType").alias("RelationshipType"),
        F.col("c.PrimaryContactEmail").alias("PrimaryContactEmail"),
        F.col("c.Owner").alias("Seller"),
        F.col("c.OwnerEmail").alias("SellerEmail"),
        F.col("c.Territory").alias("Territory"),
        F.col("c.Status").alias("AccountStatus"),
    )
    .withColumn("MatchStatus", F.when(F.col("CrmAccountID").isNotNull(), F.lit("Matched")).otherwise(F.lit("No CRM match")))
    .withColumn("CorrelatedAt", F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
)

total = correlation.count()
matched = correlation.where(F.col("MatchStatus") == "Matched").count()
print(f"   Tenants: {total} | Matched to a CRM account: {matched} | Unmatched: {total - matched}")
correlation.orderBy("MatchStatus", "TenantDomain").show(50, truncate=False)

# ── Save the correlation table ───────────────────────────────────────────────
CORR_PATH = f"{_TABLES}/inforcer_crm_correlation"
correlation.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(CORR_PATH)
print(f"✅ Saved correlation to dbo.inforcer_crm_correlation ({total} rows)")


StatementMeta(, 78055c86-4012-45a8-9810-73b0c40d77a7, 7, Finished, Available, Finished, False)

🔗 Correlating Inforcer tenants with CRM accounts by email domain...
   Tenants: 5 | Matched to a CRM account: 3 | Unmatched: 2
+----------------------+------------------------------------+--------------------+-------------------------------+------------------------------------+----------------------+----------------+-----------------------------+-------------------+------------------------------------+-----------------+-------------+------------+-------------------+
|InforcerClientTenantId|MsTenantId                          |InforcerFriendlyName|TenantDomain                   |CrmAccountID                        |CrmAccountName        |RelationshipType|PrimaryContactEmail          |Seller             |SellerEmail                         |Territory        |AccountStatus|MatchStatus |CorrelatedAt       |
+----------------------+------------------------------------+--------------------+-------------------------------+------------------------------------+----------------------+-----------

## Customer tenant domains (for CRM matching)

The **security assessment** tables only capture a tenant *display name* (`tenant_name`) — there is **no domain column**.
The tenant **domain** lives in `dbo.inforcer_tenants.tenantDnsName`.

This builds `dbo.customer_tenant_domains`: one row per assessed security-assessment tenant with a resolved **domain**, using:
1. the `tenant_name` itself when it is already a domain (e.g. `grandcereals.com`), else
2. the Inforcer domain matched on `tenantFriendlyName`.

The resolved domain is then matched to the CRM **Email (Primary Contact)** domain (`dbo.crm_accounts.PrimaryContactEmail`)
to attach the CRM account and responsible seller.


In [ ]:
# Build dbo.customer_tenant_domains: resolve a domain per security-assessment tenant, then match to CRM
from pyspark.sql import functions as F
from datetime import datetime

print("🌐 Building customer tenant domain table from security assessments...")

_WS = "0f895a7e-09c6-4645-8b47-d272bc687b8a"   # SEManagedService
_LH = "3d0144b0-12bf-4483-9508-67b26b1fd125"   # ManagedServiceData
_T = f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{_LH}/Tables/dbo"

# Regex: a bare domain (no spaces, has a dot, valid TLD) e.g. grandcereals.com, kttc.ac.ke
DOMAIN_RE = r"^[A-Za-z0-9][A-Za-z0-9.-]*\.[A-Za-z]{2,}$"

# ── 1) Distinct security-assessment tenants ──────────────────────────────────
sec = spark.read.format("delta").load(f"{_T}/security_assessment_assessments")
tenants = (
    sec.select(F.trim("tenant_name").alias("TenantName"))
       .where(F.col("TenantName").isNotNull() & (F.col("TenantName") != ""))
       .distinct()
)

# ── 2) Inforcer domain lookup (by friendly name) ─────────────────────────────
inf = (
    spark.read.format("delta").load(f"{_T}/inforcer_tenants")
         .select(
             F.lower(F.trim("tenantFriendlyName")).alias("inf_friendly"),
             F.lower(F.trim("tenantDnsName")).alias("inf_domain"),
         )
         .where(F.col("inf_friendly").isNotNull())
         .dropDuplicates(["inf_friendly"])
)

# ── 3) Resolve a domain for each tenant ──────────────────────────────────────
resolved = (
    tenants
    .withColumn("name_is_domain", F.col("TenantName").rlike(DOMAIN_RE))
    .join(inf, F.lower(F.trim(F.col("TenantName"))) == F.col("inf_friendly"), "left")
    .withColumn(
        "TenantDomain",
        F.when(F.col("name_is_domain"), F.lower(F.col("TenantName"))).otherwise(F.col("inf_domain")),
    )
    .withColumn(
        "DomainSource",
        F.when(F.col("name_is_domain"), F.lit("tenant_name_literal"))
         .when(F.col("inf_domain").isNotNull(), F.lit("inforcer_friendly_match"))
         .otherwise(F.lit("unresolved")),
    )
    .select("TenantName", "TenantDomain", "DomainSource")
    .dropDuplicates(["TenantName"])
)

# ── 4) Match resolved domain to CRM primary-contact email domain ─────────────
crm = spark.read.format("delta").load(f"{_T}/crm_accounts")
crm_dom = (
    crm.where((F.col("PrimaryContactEmail").isNotNull()) & (F.col("PrimaryContactEmail") != ""))
       .withColumn("crm_domain", F.lower(F.trim(F.element_at(F.split("PrimaryContactEmail", "@"), -1))))
       .where(F.col("crm_domain") != "")
)

final = (
    resolved.alias("d")
    .join(crm_dom.alias("c"), F.col("d.TenantDomain") == F.col("c.crm_domain"), "left")
    .select(
        F.col("d.TenantName"),
        F.col("d.TenantDomain"),
        F.col("d.DomainSource"),
        F.col("c.AccountID").alias("CrmAccountID"),
        F.col("c.Name").alias("CrmAccountName"),
        F.col("c.RelationshipType"),
        F.col("c.PrimaryContactEmail").alias("CrmPrimaryContactEmail"),
        F.col("c.Owner").alias("Seller"),
        F.col("c.OwnerEmail").alias("SellerEmail"),
        F.col("c.Territory"),
        F.col("c.Status").alias("AccountStatus"),
    )
    .withColumn("CrmMatch", F.when(F.col("CrmAccountID").isNotNull(), F.lit("Matched")).otherwise(F.lit("No CRM match")))
    .withColumn("BuiltAt", F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
    .dropDuplicates(["TenantName", "CrmAccountID"])
)

# ── 5) Report + save ─────────────────────────────────────────────────────────
total = final.select("TenantName").distinct().count()
with_domain = final.where(F.col("TenantDomain").isNotNull()).select("TenantName").distinct().count()
crm_matched = final.where(F.col("CrmMatch") == "Matched").select("TenantName").distinct().count()
print(f"   Tenants: {total} | With resolved domain: {with_domain} | Matched to CRM: {crm_matched}")

print("\n   Domain source breakdown:")
final.select("TenantName", "DomainSource").distinct().groupBy("DomainSource").count().orderBy(F.desc("count")).show(truncate=False)

print("   Tenants with a resolved domain / CRM match:")
final.where(F.col("TenantDomain").isNotNull()).orderBy("CrmMatch", "TenantName").show(50, truncate=False)

OUT_PATH = f"{_T}/customer_tenant_domains"
final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(OUT_PATH)
print(f"✅ Saved dbo.customer_tenant_domains ({final.count()} rows)")


StatementMeta(, 1f185af4-9404-410b-aaba-f29c8d185bf0, 7, Finished, Available, Finished, False)

🌐 Building customer tenant domain table from security assessments...
   Tenants: 86 | With resolved domain: 4 | Matched to CRM: 2

   Domain source breakdown:
+-----------------------+-----+
|DomainSource           |count|
+-----------------------+-----+
|unresolved             |82   |
|inforcer_friendly_match|2    |
|tenant_name_literal    |2    |
+-----------------------+-----+

   Tenants with a resolved domain / CRM match:
+----------------+--------------------+-----------------------+------------------------------------+--------------+----------------+-----------------------------+-------------+-----------------------------+----------------+-------------+------------+-------------------+
|TenantName      |TenantDomain        |DomainSource           |CrmAccountID                        |CrmAccountName|RelationshipType|CrmPrimaryContactEmail       |Seller       |SellerEmail                  |Territory       |AccountStatus|CrmMatch    |BuiltAt            |
+----------------+---------